Code-only archive: embedded input/reference bundles were omitted. Run the adjacent build_notebook.py or build_notebooks.py with your local inputs to recreate the executable Kaggle notebook.


# PROTEUS classifier comparison: verified report

CPU-only verification of the70 saved evaluations from training notebook version4. All training-source hashes,baseline labels,official classification metrics,probability metrics and frozen CV selection are checked. This notebook also reports descriptive selection stability. It does not refit models. The original training notebook has a final-report import conflict corrected here.

In [ ]:
import os,sys,pathlib,json,hashlib,shutil,subprocess,base64
os.environ.update(OPENBLAS_NUM_THREADS="1",OMP_NUM_THREADS="1",MKL_NUM_THREADS="1",MPLCONFIGDIR="/tmp/classifier-report-mpl")
selections=list(pathlib.Path("/kaggle/input").rglob("selection.json"))
assert len(selections)==1,selections
previous=selections[0].parent;root=previous.parent
source=root/"riemann-project"
manifest=(source/"source-integrity.json").read_bytes()
assert hashlib.sha256(manifest).hexdigest()=="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81"
for rel,digest in json.loads(manifest).items():
    assert hashlib.sha256((source/rel).read_bytes()).hexdigest()==digest,rel
project=pathlib.Path("/kaggle/working/verified-project")
shutil.copytree(source,project,ignore=shutil.ignore_patterns("data","cache","__pycache__"))
experiment=project/"reproduction/classifier_ablation"
(experiment/"analyze.py").write_bytes(base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text()))
(experiment/"test_analysis.py").write_bytes(base64.b64decode(__import__("pathlib").Path("local-bundle.base64").read_text()))
out=pathlib.Path("/kaggle/working/classifier-results")
shutil.copytree(previous,out)
envdir=pathlib.Path("/tmp/classifier-report-env")
subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(envdir)],check=True)
python=str(envdir/"bin/python")
pins=["numpy==2.5.4","scipy==1.18.1","scikit-learn==1.9.1","pyriemann==0.12","pandas==3.1.0","threadpoolctl==3.7.0"]
subprocess.run([sys.executable,"-m","pip","--python",python,"install","--disable-pip-version-check","--quiet",*pins],check=True)
env=dict(os.environ,CLASSIFIER_OUTPUT=str(out))
subprocess.run([python,"-c","import analyze;assert analyze.config_hash()=='12abff9e82c4954355840023e94af27c90981efd953793259bbd59f8eca2e827'"],cwd=experiment,env=env,check=True)
subprocess.run([python,"-m","pytest","-q","test_analysis.py","--tb=short"],cwd=experiment,env=env,check=True)
(out/"analysis-provenance.json").write_text(json.dumps(dict(training_source="den1shh/proteus-lag4-classifier-ablation",training_version=4,classifier_config_sha256="12abff9e82c4954355840023e94af27c90981efd953793259bbd59f8eca2e827",original_source_manifest_sha256="3e8cf0b952a1a18ab496a89d6a31275522e8efa003ea6b8ed4b9e6467d2a3c81",analyzer_sha256=hashlib.sha256((experiment/"analyze.py").read_bytes()).hexdigest(),refitted_models=0),indent=2))
print("Original source hashes verified; unchanged predictions copied; analysis tests passed",flush=True)


In [ ]:
subprocess.run([python,"-u",str(experiment/"analyze.py")],cwd=experiment,env=env,check=True)
verification=json.loads((out/"verification.json").read_text())
assert verification["verified_jobs"]==70 and verification["exact_baseline_prediction_jobs"]==10
assert verification["independently_recomputed_log_loss_and_brier"]
print((out/"REPORT.md").read_text())
print("All70 saved evaluations independently verified. No model refits.")
